In [1]:
import rospy
import actionlib
import ipywidgets as widgets
from IPython.display import display, clear_output
from geometry_msgs.msg import Twist
from nav_msgs.msg import Odometry
from assignment_2_2024.msg import PlanningAction, PlanningGoal
from rt1_assignment2_p1.msg import RobotStatus
from rt1_assignment2_p1.srv import GetLastTarget

In [2]:
rospy.init_node('jupyter_action_client_node', anonymous=True)
client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
client.wait_for_server()
status_pub = rospy.Publisher('/robot_status', RobotStatus, queue_size=10)
current_feedback = {'x': 0.0, 'y': 0.0, 'status': ""}

rospy.loginfo("Jupyter ROS node initialized and action client is ready.")


[INFO] [1743968959.977089, 1237.078000]: Jupyter ROS node initialized and action client is ready.


In [3]:
def odom_callback(msg: Odometry):
    position = msg.pose.pose.position
    velocity = msg.twist.twist

    status_msg = RobotStatus()
    status_msg.x = position.x
    status_msg.y = position.y
    status_msg.vel_x = velocity.linear.x
    status_msg.vel_z = velocity.angular.z

    status_pub.publish(status_msg)

rospy.Subscriber('/odom', Odometry, odom_callback)


In [4]:
def send_goal(x: float, y: float) -> None:
    goal = PlanningGoal()
    goal.target_pose.pose.position.x = x
    goal.target_pose.pose.position.y = y
    client.send_goal(goal, done_cb=goal_done_callback, feedback_cb=goal_feedback_callback)
    set_last_target(x, y)
    with log_output:
        print(f"Sent goal: x={x}, y={y}")

def cancel_goal() -> None:
    client.cancel_goal()
    with log_output:
        print("Goal canceled.")

def goal_done_callback(status, result):
    with log_output:
        if status == actionlib.GoalStatus.SUCCEEDED:
            print("Goal achieved successfully.")
        else:
            print("Goal did not complete successfully.")

def goal_feedback_callback(feedback):
    current_feedback['x'] = feedback.actual_pose.position.x
    current_feedback['y'] = feedback.actual_pose.position.y
    current_feedback['status'] = feedback.stat

def get_last_target():
    rospy.wait_for_service('/get_last_target')
    try:
        proxy = rospy.ServiceProxy('/get_last_target', GetLastTarget)
        resp = proxy(False, 0, 0)
        with log_output:
            print(f"Last target: x={resp.res_x}, y={resp.res_y}")
    except rospy.ServiceException as e:
        with log_output:
            print(f"Service call failed: {e}")

def set_last_target(x: float, y: float) -> None:
    rospy.wait_for_service('/get_last_target')
    try:
        proxy = rospy.ServiceProxy('/get_last_target', GetLastTarget)
        resp = proxy(True, x, y)
        if resp.success:
            with log_output:
                print("Last target successfully updated.")
    except rospy.ServiceException as e:
        with log_output:
            print(f"Service call failed: {e}")


In [5]:
x_input = widgets.FloatText(description="X:", value=0.0)
y_input = widgets.FloatText(description="Y:", value=0.0)
send_btn = widgets.Button(description="Send Goal")
cancel_btn = widgets.Button(description="Cancel")
status_btn = widgets.Button(description="Status")
feedback_btn = widgets.Button(description="Feedback")
last_btn = widgets.Button(description="Last Target")
clear_btn = widgets.Button(description="Clear Log")
log_output = widgets.Output(layout={"border": "1px solid black", "height": "200px", "overflow_y": "scroll"})


In [6]:
def on_send_clicked(_):
    if client.get_state() == actionlib.GoalStatus.ACTIVE:
        with log_output:
            print("A goal is currently active. Cancel it first.")
    else:
        send_goal(x_input.value, y_input.value)

def on_cancel_clicked(_):
    cancel_goal()

def on_status_clicked(_):
    if client.get_state() == actionlib.GoalStatus.ACTIVE:
        with log_output:
            print(f"Current status: {current_feedback['status']}")
    else:
        with log_output:
            print("No active goal.")

def on_feedback_clicked(_):
    if client.get_state() == actionlib.GoalStatus.ACTIVE:
        x = current_feedback['x']
        y = current_feedback['y']
        with log_output:
            print(f"Current feedback position: x={x}, y={y}")
    else:
        with log_output:
            print("No active goal.")

def on_last_clicked(_):
    get_last_target()

def on_clear_clicked(_):
    log_output.clear_output()


In [7]:
send_btn.on_click(on_send_clicked)
cancel_btn.on_click(on_cancel_clicked)
status_btn.on_click(on_status_clicked)
feedback_btn.on_click(on_feedback_clicked)
last_btn.on_click(on_last_clicked)
clear_btn.on_click(on_clear_clicked)

controls = widgets.VBox([
    widgets.HBox([x_input, y_input, send_btn]),
    widgets.HBox([cancel_btn, status_btn, feedback_btn, last_btn, clear_btn]),
    log_output
])

display(controls)


Widget Javascript not detected.  It may not be installed or enabled properly.
